# CX Assist : Notebook 04: Dynamic Case Investigation

This notebook combines structured S3 data, policy retrieval, a configurable LLM provider, and strict Pydantic schemas to investigate any selected case. The model receives only supplied evidence, recommends but never executes an action, and always requires human review.


## 1. Imports and configuration


In [1]:
from __future__ import annotations

import io
import json
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from typing import Any, Literal

import boto3
import pandas as pd
from dotenv import load_dotenv
from langchain_core.language_models.chat_models import BaseChatModel
from pydantic import BaseModel, Field, model_validator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv(override=True)

AWS_PROFILE = os.getenv("AWS_PROFILE", "CXASSIST")
AWS_REGION = os.getenv("AWS_REGION", "ap-south-1")
S3_BUCKET = os.getenv("S3_BUCKET", "rahulcxassistdemo")
S3_PREFIX = os.getenv("S3_PREFIX", "cx-copilot/dev").strip("/")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "groq").strip().lower()
MODEL_TEMPERATURE = float(os.getenv("MODEL_TEMPERATURE", "0"))

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")
def require_setting(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"Required environment variable '{name}' is missing.")
    return value

def get_chat_model(provider: str | None = None) -> BaseChatModel:
    selected = (provider or MODEL_PROVIDER).strip().lower()
    if selected == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(
            model=require_setting("GROQ_MODEL"),
            api_key=require_setting("GROQ_API_KEY"),
            temperature=MODEL_TEMPERATURE, max_retries=3, timeout=90,
        )
    if selected == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(
            model=require_setting("GOOGLE_MODEL"),
            google_api_key=require_setting("GOOGLE_API_KEY"),
            temperature=MODEL_TEMPERATURE, max_retries=4, timeout=90,
        )
    if selected == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model=require_setting("OPENAI_MODEL"),
            api_key=require_setting("OPENAI_API_KEY"),
            temperature=MODEL_TEMPERATURE, max_retries=3, timeout=90,
        )
    raise ValueError(f"Unsupported MODEL_PROVIDER '{selected}'.")

chat_model = get_chat_model()
active_model_name = os.getenv(f"{MODEL_PROVIDER.upper()}_MODEL", "Unknown")
print(f"Ready: provider={MODEL_PROVIDER}, model={active_model_name}")


Ready: provider=groq, model=openai/gpt-oss-120b


## 2. Load structured datasets and policies from S3


In [2]:
DATASET_FILES = [
    "customers", "vehicles", "contracts", "invoices",
    "payments", "service_records", "cases", "case_interactions",
]

def s3_bytes(relative_path: str) -> bytes:
    key = f"{S3_PREFIX}/{relative_path.lstrip('/')}"
    return s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()

datasets = {
    name: pd.read_csv(io.BytesIO(s3_bytes(f"structured/{name}.csv")))
    for name in DATASET_FILES
}

policy_prefix = f"{S3_PREFIX}/policies/"
policy_listing = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=policy_prefix)
policy_documents = {}
for item in policy_listing.get("Contents", []):
    key = item["Key"]
    if key.endswith(".md"):
        policy_documents[key.rsplit("/", 1)[-1]] = (
            s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read().decode("utf-8")
        )

print(f"Loaded {len(datasets)} datasets and {len(policy_documents)} policies.")


Loaded 8 datasets and 4 policies.


## 3. Generic Case 360 service


In [3]:
class CaseNotFoundError(ValueError):
    pass

def records(frame: pd.DataFrame, column: str, value: Any) -> list[dict]:
    if value is None or pd.isna(value):
        return []
    subset = frame.loc[frame[column] == value]
    return subset.where(pd.notna(subset), None).to_dict(orient="records")

def one(frame: pd.DataFrame, column: str, value: Any) -> dict | None:
    found = records(frame, column, value)
    return found[0] if found else None

def get_case_context(case_id: str) -> dict:
    normalized = case_id.strip().upper()
    case = one(datasets["cases"], "case_id", normalized)
    if case is None:
        raise CaseNotFoundError(f"Case '{normalized}' was not found.")
    customer_id = case.get("customer_id")
    unit_number = case.get("unit_number")
    invoice_id = case.get("invoice_id")
    invoice = one(datasets["invoices"], "invoice_id", invoice_id)
    related_invoice_id = invoice.get("related_invoice_id") if invoice else None
    related_invoice = None
    if related_invoice_id and not pd.isna(related_invoice_id):
        related_invoice = one(
            datasets["invoices"], "invoice_id", related_invoice_id
        )
    contract_id = invoice.get("contract_id") if invoice else None
    invoice_ids = [
        current_id for current_id in [invoice_id, related_invoice_id]
        if current_id and not pd.isna(current_id)
    ]
    invoice_subset = datasets["invoices"].loc[
        datasets["invoices"]["invoice_id"].isin(invoice_ids)
    ]
    payment_subset = datasets["payments"].loc[
        datasets["payments"]["invoice_id"].isin(invoice_ids)
    ]
    return {
        "case": case,
        "customer": one(datasets["customers"], "customer_id", customer_id),
        "vehicle": one(datasets["vehicles"], "unit_number", unit_number),
        "contract": one(datasets["contracts"], "contract_id", contract_id),
        "invoice": invoice,
        "related_invoice": related_invoice,
        "invoices_under_review": invoice_subset.where(
            pd.notna(invoice_subset), None
        ).to_dict(orient="records"),
        "payments": payment_subset.where(
            pd.notna(payment_subset), None
        ).to_dict(orient="records"),
        "service_records": records(datasets["service_records"], "unit_number", unit_number),
        "interactions": records(datasets["case_interactions"], "case_id", normalized),
    }


## 4. Citation-aware policy retriever


In [4]:
@dataclass(frozen=True)
class PolicyChunk:
    document_id: str
    document_name: str
    version: str
    section: str
    content: str
    source_key: str

def meta(text: str, label: str) -> str:
    match = re.search(rf"\*\*{re.escape(label)}:\*\*\s*(.+)", text)
    return match.group(1).strip() if match else "Unknown"

def split_policy(filename: str, text: str) -> list[PolicyChunk]:
    title_match = re.search(r"^#\s+(.+)$", text, re.MULTILINE)
    title = title_match.group(1).strip() if title_match else filename
    document_id, version = meta(text, "Document ID"), meta(text, "Version")
    output = []
    for raw in re.split(r"^##\s+", text, flags=re.MULTILINE)[1:]:
        lines = raw.strip().splitlines()
        if len(lines) > 1:
            output.append(PolicyChunk(
                document_id, title, version, lines[0].strip(),
                "\n".join(lines[1:]).strip(),
                f"{S3_PREFIX}/policies/{filename}",
            ))
    return output

chunks = [c for f, t in policy_documents.items() for c in split_policy(f, t)]
search_text = [f"{c.document_name} {c.section} {c.content}" for c in chunks]
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
policy_matrix = vectorizer.fit_transform(search_text)

POLICY_HINTS = {
    "Billing Dispute": {"POL-BILL-001", "POL-SVC-003"},
    "Duplicate Charge": {"POL-BILL-001", "POL-BILL-002"},
    "Billing Explanation": {"POL-BILL-001"},
    "Payment Reconciliation": {"POL-BILL-001", "POL-CX-004"},
    "Roadside Assistance": {"POL-CX-004"},
}

def retrieve_policies(context: dict, top_k: int = 5) -> list[dict]:
    case = context["case"]
    query = " ".join(str(case.get(k, "")) for k in (
        "issue_type", "priority", "subject", "description"
    ))
    scores = cosine_similarity(vectorizer.transform([query]), policy_matrix).flatten()
    hints = POLICY_HINTS.get(case["issue_type"], set())
    ranked = []
    for i, chunk in enumerate(chunks):
        item = asdict(chunk)
        item["retrieval_score"] = round(float(scores[i]) + (0.15 if chunk.document_id in hints else 0), 4)
        item["citation"] = f"{chunk.document_id} v{chunk.version} — {chunk.section}"
        ranked.append(item)
    return sorted(ranked, key=lambda x: x["retrieval_score"], reverse=True)[:top_k]


## 5. Structured investigation contract


In [5]:
class EvidenceItem(BaseModel):
    source_system: str
    reference_id: str
    fact: str

class PolicyCitation(BaseModel):
    document_id: str
    version: str
    section: str
    relevance: str

class InvestigationResult(BaseModel):
    case_id: str
    issue_type: str
    case_summary: str
    verified_facts: list[str]
    evidence: list[EvidenceItem]
    policy_citations: list[PolicyCitation]
    missing_information: list[str]
    recommended_action: str
    approval_level: Literal["CX Agent", "CX Supervisor", "CX Manager", "Billing Operations", "Fleet Maintenance"]
    confidence: float = Field(ge=0, le=1)
    requires_human_review: bool
    customer_response_draft: str

    @model_validator(mode="after")
    def enforce_human_review(self):
        if not self.requires_human_review:
            raise ValueError("Every recommendation requires human review.")
        return self


## 6. Investigation prompt and execution


In [6]:
SYSTEM_PROMPT = """
You are an internal CX investigation assistant. Analyze only the supplied operational records and policy excerpts.

Rules:
1. Never invent a fact, identifier, policy, calculation, or customer commitment.
2. Every evidence item must use an identifier present in the supplied records.
3. Every policy citation must use a supplied document ID, version, and section.
4. Put absent or conflicting facts in missing_information.
5. Recommendations are decision support only. Never claim that a refund, credit, closure, or escalation has been completed.
6. requires_human_review must always be true.
7. The customer draft must be professional and must describe unapproved actions as pending review.
8. Confidence must reflect evidence completeness, not writing confidence.
9. Each verified fact must be a separate, specific statement supported by a supplied record.
10. Include only evidence relevant to determining the issue and recommended action. Do not cite unrelated records merely because they were retrieved.
11. For duplicate-charge cases, compare the disputed invoice with its related original invoice and compare their corresponding payment transactions.
12. Do not mark information as missing when it appears anywhere in the supplied operational records.
13. Evidence reference_id values must be selected only from allowed_reference_ids.
14. Never create placeholder identifiers such as NONE, N/A, UNKNOWN, PAYMENTS-NONE, or RECORD-NOT-FOUND.
15. The absence of a record is not an EvidenceItem. Describe it only in missing_information.
""".strip()

def extract_reference_ids(context: dict) -> set[str]:
    reference_ids = set()
    for value in context.values():
        items = value if isinstance(value, list) else [value]
        for item in items:
            if not isinstance(item, dict):
                continue
            for key, item_value in item.items():
                is_reference = key.endswith("_id") or key == "unit_number"
                if is_reference and item_value and not pd.isna(item_value):
                    reference_ids.add(str(item_value))
    return reference_ids

def invoke_with_retry(structured_model, prompt: str, max_attempts: int = 5):
    """Retry temporary provider errors with exponential backoff and jitter."""
    last_error = None
    transient_markers = (
        "503", "unavailable", "high demand",
        "temporarily unavailable", "timeout", "retry in",
    )
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"Model invocation attempt {attempt}/{max_attempts}")
            return structured_model.invoke(prompt)
        except Exception as exc:
            last_error = exc
            message = str(exc).lower()
            daily_quota_exhausted = any(marker in message for marker in (
                "free_tier_requests", "requestsperday",
                "perdayperproject", "daily limit",
            ))
            if daily_quota_exhausted:
                raise RuntimeError(
                    "The active provider's daily quota is exhausted. "
                    "Switch MODEL_PROVIDER or enable provider billing."
                ) from exc
            if not any(marker in message for marker in transient_markers):
                raise
            if attempt == max_attempts:
                break
            delay = min(2 ** attempt + random.uniform(0, 1), 20)
            print(f"Temporary provider error. Retrying in {delay:.1f} seconds...")
            time.sleep(delay)
    raise RuntimeError(
        f"Model remained unavailable after {max_attempts} attempts. "
        f"Last error: {last_error}"
    ) from last_error


def investigate_case(case_id: str) -> tuple[InvestigationResult, dict, list[dict], float]:
    context = get_case_context(case_id)
    policies = retrieve_policies(context)
    allowed_reference_ids = sorted(extract_reference_ids(context))
    allowed_policy_ids = sorted({p["document_id"] for p in policies})
    payload = {
        "operational_records": context,
        "retrieved_policy_sections": policies,
        "allowed_reference_ids": allowed_reference_ids,
        "allowed_policy_ids": allowed_policy_ids,
    }
    prompt = (
        f"{SYSTEM_PROMPT}\n\nINVESTIGATION INPUT:\n"
        + json.dumps(payload, indent=2, default=str)
    )
    structured_model = chat_model.with_structured_output(InvestigationResult)
    started = time.perf_counter()
    result = invoke_with_retry(
        structured_model=structured_model, prompt=prompt, max_attempts=5
    )
    latency = round(time.perf_counter() - started, 3)
    return result, context, policies, latency


## 7. Investigate any selected case


In [10]:
available_case_ids = sorted(datasets["cases"]["case_id"].tolist())
selected_case_id="CASE-3021"
#selected_case_id = available_case_ids[0]  # Change to any listed case ID.

investigation, case_context, retrieved_policies, latency_seconds = investigate_case(selected_case_id)
print(f"Completed {selected_case_id} in {latency_seconds} seconds.")
display(investigation)


Model invocation attempt 1/5
Completed CASE-3021 in 4.865 seconds.


InvestigationResult(case_id='CASE-3021', issue_type='Billing Dispute', case_summary='Customer disputes full monthly lease charge for August 2026, citing 78 hours of vehicle downtime due to an unscheduled repair.', verified_facts=['CASE-3021 was opened on 2026-09-03 by Northstar Retail Logistics disputing invoice INV-1047.', 'UNIT-4521 downtime due to unscheduled repair lasted 78 hours from 2026-08-10 to 2026-08-13.', 'Invoice INV-1047 amount is $4800.00 for the August 2026 billing period.', 'Contract CTR-2001 monthly lease rate is $4800.00 and was active during the disputed period.', 'Customer requested credit for three days of downtime in interaction INT-8001.', 'Agent confirmed billing review pending in interaction INT-8002.'], evidence=[EvidenceItem(source_system='case', reference_id='CASE-3021', fact='Customer opened case CASE-3021 disputing the full monthly lease charge for August 2026 due to vehicle downtime.'), EvidenceItem(source_system='service_records', reference_id='SRV-7001

## 8. Deterministic grounding validation


In [11]:
def validate_and_clean_grounding(
    result: InvestigationResult, context: dict, policies: list[dict]
) -> tuple[InvestigationResult, dict]:
    expected_case_id = context["case"]["case_id"]
    allowed_reference_ids = extract_reference_ids(context)
    allowed_policy_ids = {p["document_id"] for p in policies}
    invalid_evidence = [
        e.reference_id for e in result.evidence
        if e.reference_id not in allowed_reference_ids
    ]
    invalid_policy_ids = [
        p.document_id for p in result.policy_citations
        if p.document_id not in allowed_policy_ids
    ]
    result.evidence = [
        e for e in result.evidence if e.reference_id in allowed_reference_ids
    ]
    result.policy_citations = [
        p for p in result.policy_citations if p.document_id in allowed_policy_ids
    ]
    checks = {
        "case_id_matches": result.case_id == expected_case_id,
        "human_review_required": result.requires_human_review is True,
        "has_valid_evidence": len(result.evidence) > 0,
        "unsupported_evidence_removed": invalid_evidence,
        "unsupported_policies_removed": invalid_policy_ids,
        "evidence_ids_grounded": len(invalid_evidence) == 0,
        "policy_ids_grounded": len(invalid_policy_ids) == 0,
    }
    critical_checks = [
        checks["case_id_matches"], checks["human_review_required"],
        checks["has_valid_evidence"],
    ]
    if not all(critical_checks):
        raise ValueError({
            "message": "Investigation failed critical grounding validation.",
            "checks": checks,
        })
    return result, checks

investigation, grounding_checks = validate_and_clean_grounding(
    investigation, case_context, retrieved_policies
)
display(grounding_checks)
display(investigation)


{'case_id_matches': True,
 'human_review_required': True,
 'has_valid_evidence': True,
 'unsupported_evidence_removed': [],
 'unsupported_policies_removed': [],
 'evidence_ids_grounded': True,
 'policy_ids_grounded': True}

InvestigationResult(case_id='CASE-3021', issue_type='Billing Dispute', case_summary='Customer disputes full monthly lease charge for August 2026, citing 78 hours of vehicle downtime due to an unscheduled repair.', verified_facts=['CASE-3021 was opened on 2026-09-03 by Northstar Retail Logistics disputing invoice INV-1047.', 'UNIT-4521 downtime due to unscheduled repair lasted 78 hours from 2026-08-10 to 2026-08-13.', 'Invoice INV-1047 amount is $4800.00 for the August 2026 billing period.', 'Contract CTR-2001 monthly lease rate is $4800.00 and was active during the disputed period.', 'Customer requested credit for three days of downtime in interaction INT-8001.', 'Agent confirmed billing review pending in interaction INT-8002.'], evidence=[EvidenceItem(source_system='case', reference_id='CASE-3021', fact='Customer opened case CASE-3021 disputing the full monthly lease charge for August 2026 due to vehicle downtime.'), EvidenceItem(source_system='service_records', reference_id='SRV-7001

## 9. Optional multi-case run

This makes one paid model call per case. Run it only after the selected-case test passes.


In [12]:
RUN_ALL_CASES = False
evaluation_rows = []

if RUN_ALL_CASES:
    for case_id in available_case_ids:
        result, context, policies, latency = investigate_case(case_id)
        result, checks = validate_and_clean_grounding(result, context, policies)
        evaluation_rows.append({
            "case_id": case_id, "issue_type": result.issue_type,
            "confidence": result.confidence, "latency_seconds": latency,
            "evidence_count": len(result.evidence),
            "policy_count": len(result.policy_citations),
            "grounding_passed": all([
                checks["case_id_matches"],
                checks["human_review_required"],
                checks["has_valid_evidence"],
                checks["evidence_ids_grounded"],
                checks["policy_ids_grounded"],
            ]),
        })
    display(pd.DataFrame(evaluation_rows))
else:
    print("Skipped. Set RUN_ALL_CASES=True to evaluate every case.")


Skipped. Set RUN_ALL_CASES=True to evaluate every case.


## Completion criteria

Notebook 04 is complete when any selected case produces a valid InvestigationResult, human review is true, evidence IDs exist in the Case 360 context, and cited policy IDs exist in the retrieved policy set. Notebook 05 will move these components into LangGraph nodes and add workflow state, conditional routing, and a human-review checkpoint.
